# Lam2016

[https://doi.org/10.1038/s41467-026-70414-2](https://doi.org/10.1038/s41467-026-70414-2)

In [ ]:
import string

import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib import gridspec
from mxlpy import Model, Simulator, make_protocol, scan
from mxlpy.parallel import parallelise

from mxlmodels import get_lam2026


## Helper Functions

In [ ]:
def create_npq1(model: Model):
    model.update_parameters({
        "tau_0": 1.693954731,
        "kappa_qZ": 0,
        "kappa_QA": 0,
        "kappa_QZ": 0,
        "V_tot_WT": 49.8
    })
    
    for reac in ["VA", "AV", "AZ", "PAf", "PAb", "PZf", "PZb", "QAb", "QAf", "QZf", "QZb"]:
        if reac in model.get_reaction_names():
            model.remove_reaction(reac)
    
    return model

def create_npq4(model: Model):
    model.update_parameters({
        "tau_0": 1.702050733,
        "V_tot_WT": 40.6,
        "kappa_QV": 0,
        "kappa_QA": 0,
        # "kappa_QZ": 0,
        "kappa_QL": 0,
    })
    
    for reac in ["QVf", "QAb", "QAf", "QZf", "QZb"]:
        if reac in model.get_reaction_names():
            model.remove_reaction(reac)
    
    return model

def create_npq4npq1(model: Model):
    model.update_parameters({
        "tau_0": 1.283586052,
        "kappa_QV": 0,
        "kappa_QA": 0,
        "kappa_QZ": 0,
        "kappa_QL": 0,
        "kappa_qZ": 0,
        "kappa_qI": 7.05,
    })
    
    return model

def create_lut2(model: Model):
    model.update_parameters({
        "tau_0": 1.453352343,
        "V_tot_WT": 71.2,
        "kappa_QL": 0,
        "k_PV_f": 1.43,
        "k_PV_b": 13.1,
        "k_PA_f": 34.4,
        "k_PA_b": 294,
        "k_PZ_f": 74.1,
        "k_PZ_b": 168,
        "P_tot": 49.9,
    })
    
    model.update_variable("PL", 0)
    
    return model

def create_zep2(model: Model):
    model.update_parameters({
        "tau_0": 1.453352343,
        "V_tot_WT": 10.7,
        "kappa_QL": 0,
        "k_AV": 0.006,
        "k_ZA": 0.038,
    })
    
    model.update_variable("PL", 152.7)
    
    return model

def create_npq1lut2(model: Model):
    model.update_parameters({
        "tau_0": 1.453352343,
        "V_tot_WT": 49.8,
        "k_PV_f": 1.43,
        "k_PV_b": 13.1,
        "k_PA_f": 34.4,
        "k_PA_b": 294,
        "k_PZ_f": 74.1,
        "k_PZ_b": 168,
        "kappa_qI": 7.04
    })
    
    for reac in ["VA", "AV", "AZ", "ZA", "PAf", "PAb", "PZf", "PZb", "QAb", "QAf", "QZf", "QZb"]:
        if reac in model.get_reaction_names():
            model.remove_reaction(reac)
            
    model.update_variable("PL", 49.9)
    
    return model

def create_zep2lut2(model: Model):
    model.update_parameters({
        "tau_0": 1.453352343,
        "V_tot_WT": 10.7,
        "k_PV_f": 1.43,
        "k_PV_b": 13.1,
        "k_PA_f": 34.4,
        "k_PA_b": 294,
        "k_PZ_f": 74.1,
        "k_PZ_b": 168,
        "kappa_qI": 7.04,
    })

    model.update_variable("PL", 49.9)

    return model

def create_npq4lut2(model: Model):
    model.update_parameters({
        "tau_0": 1.453352343,
        "V_tot_WT": 40.6,
        "kappa_QV": 0,
        "kappa_QA": 0,
        "kappa_QL": 0,
        "k_PV_f": 1.43,
        "k_PV_b": 13.1,
        "k_PA_f": 34.4,
        "k_PA_b": 294,
        "k_PZ_f": 74.1,
        "k_PZ_b": 168,
        "kappa_qI": 7.04,
    })
    
    for reac in ["QVf", "QVb", "QAb", "QAf", "QZf", "QZb"]:
        if reac in model.get_reaction_names():
            model.remove_reaction(reac)

    model.update_variable("PL", 49.9)

    return model

def create_npq4zep2(model: Model):
    model.update_parameters({
        "tau_0": 1.453352343,
        "V_tot_WT": 10.7,
        "kappa_QV": 0,
        "kappa_QA": 0,
        "kappa_QL": 0,
        "kappa_qI": 7.04,
    })
    
    for reac in ["QVf", "QVb", "QAb", "QAf", "QZf", "QZb"]:
        if reac in model.get_reaction_names():
            model.remove_reaction(reac)

    return model

## Figure 2

In [ ]:
def fig2_sim(settings):
    if "model" in settings:
        m = settings["model"]
    else:
        m = get_lam2026()
        m.update_parameters(settings["params"])
        m.update_variables(settings["y0"])
    
        if "remove_reacs" in settings:
            for reac in settings["remove_reacs"]:
                m.remove_reaction(reac)

    res = Simulator(m).simulate_protocol(settings["protocol"], time_points_per_step=int(1e3)).get_result().unwrap_or_err().get_combined()
    res.index = res.index - settings["protocol"].index[0].total_seconds()
    return res[res.index >= 0]
    

fig2_prtc = make_protocol([
    (1, {"ppfd": 0}),
    (5, {"ppfd": 1}),
    (10, {"ppfd": 0}),
    (5, {"ppfd": 1}),
])

fig2_settings = {
    "npq4npq1": {
        "model": create_npq4npq1(get_lam2026()),
        "protocol": fig2_prtc
    },
    "npq4": {
        "model": create_npq4(get_lam2026()),
        "protocol": fig2_prtc
    },
    "npq1": {
        "model": create_npq1(get_lam2026()),
        "protocol": fig2_prtc
    },
    "lut2": {
        "model": create_lut2(get_lam2026()),
        "protocol": fig2_prtc
    },
    "zep2": {
        "model": create_zep2(get_lam2026()),
        "protocol": fig2_prtc
    }
}

fig2_res = dict(parallelise(fig2_sim, list(fig2_settings.items())))
fig2_res = pd.concat(fig2_res.values(), keys=fig2_res.keys())
fig2_res.index.names = ["GT", "time"]

In [ ]:
fig2 = plt.figure(figsize=(10, 6))

gs = gridspec.GridSpec(2, 6, figure=fig2)

ax1 = fig2.add_subplot(gs[0, 0:2])
ax2 = fig2.add_subplot(gs[0, 2:4])
ax3 = fig2.add_subplot(gs[0, 4:6])
ax4 = fig2.add_subplot(gs[1, 1:3])
ax5 = fig2.add_subplot(gs[1, 3:5])

axes_settings = {
    "npq4npq1": {
        "label": "A",
        "ax": ax1
    },
    "npq4": {
        "label": "B",
        "ax": ax2
    },
    "npq1": {
        "label": "C",
        "ax": ax3
    },
    "lut2": {
        "label": "D",
        "ax": ax4
    },
    "zep2": {
        "label": "E",
        "ax": ax5
    },
}

box = fig2_prtc.copy()
box.index = box.index.total_seconds()
box.index = box.index - box.index[0]

for GT, df in fig2_res.groupby(level="GT"):
    res = df.droplevel("GT")
    
    ax = axes_settings[GT]["ax"]
    ax.plot(res["tau_Fluo"], lw=3, color="#fa5555")
    
    ax.set_ylim(0, 2)
    ax.set_yticks(np.linspace(0, 1.5, 4))
    ax.set_ylabel(r"$\tau_\mathrm{avg}$")
    
    ax.set_xlim(-1, 20)
    
    ax.set_title(GT, fontweight="bold", fontstyle="italic")
    ax.text(-0.01, 1.01, axes_settings[GT]["label"], fontweight="bold", transform=ax.transAxes, ha="right", va="bottom", fontsize=12)
    
    for i, (x2, ppfd) in enumerate(box.iterrows()):
        x = ax.get_xlim()[0] if x2 == 0 else box.iloc[i - 1].name
        color = "white" if ppfd.values == 1 else "black"
        
        ax.add_patch(plt.Rectangle(
            (x, 0),
            width=x2 - x,
            height=0.1,
            facecolor=color,
            edgecolor="black"
        ))

plt.tight_layout()

## Figure 3

In [ ]:
def fig3_sim(settings):
    res = Simulator(settings["model"]).simulate_protocol(settings["protocol"], time_points_per_step=int(1e3)).get_result().unwrap_or_err().get_combined()
    
    res.index = res.index - settings["protocol"].index[0].total_seconds()
    return res[res.index >= 0]
    

fig3_prtc = make_protocol([
    (1, {"ppfd": 0}),
    (5, {"ppfd": 1}),
    (10, {"ppfd": 0}),
    (5, {"ppfd": 1}),
])

fig3_prtc_base = make_protocol([(1, {"ppfd": 0}), (20, {"ppfd": 1})])
fig3_prtc_mix = make_protocol([
    (1, {"ppfd": 0}),
    (3, {"ppfd": 1}),
    (1, {"ppfd": 0}),
    (1, {"ppfd": 1}),
    (3, {"ppfd": 0}),
    (9, {"ppfd": 1}),
    (3, {"ppfd": 0})
])

fig3_settings = {
    "WT": {
        "model": get_lam2026(),
        "protocol": fig3_prtc
    },
    "WT_base": {
        "model": get_lam2026(),
        "protocol": fig3_prtc_base
    },
    "WT_mix": {
        "model": get_lam2026(),
        "protocol": fig3_prtc_mix
    },
    "zep2lut2": {
        "model": create_zep2lut2(get_lam2026()),
        "protocol": fig3_prtc
    },
    "npq1lut2": {
        "model": create_npq1lut2(get_lam2026()),
        "protocol": fig3_prtc
    },
    "npq4lut2": {
        "model": create_npq4lut2(get_lam2026()),
        "protocol": fig3_prtc
    },
    "npq4zep2": {
        "model": create_npq4zep2(get_lam2026()),
        "protocol": fig3_prtc
    },

}

fig3_res = dict(parallelise(fig3_sim, list(fig3_settings.items())))
fig3_res = pd.concat(fig3_res.values(), keys=fig3_res.keys())
fig3_res.index.names = ["GT", "time"]

In [ ]:
fig3 = plt.figure(figsize=(12, 6))

gs = gridspec.GridSpec(2, 8, figure=fig3)

ax1 = fig3.add_subplot(gs[0, 1:3])
ax2 = fig3.add_subplot(gs[0, 3:5])
ax3 = fig3.add_subplot(gs[0, 5:7])
ax4 = fig3.add_subplot(gs[1, 0:2])
ax5 = fig3.add_subplot(gs[1, 2:4])
ax6 = fig3.add_subplot(gs[1, 4:6])
ax7 = fig3.add_subplot(gs[1, 6:8])

normal_box = fig3_prtc.copy()
normal_box.index = normal_box.index.total_seconds()
normal_box.index = normal_box.index - normal_box.index[0]

base_box = fig3_prtc_base.copy()
base_box.index = base_box.index.total_seconds()
base_box.index = base_box.index - base_box.index[0]

mix_box = fig3_prtc_mix.copy()
mix_box.index = mix_box.index.total_seconds()
mix_box.index = mix_box.index - mix_box.index[0]

axes_settings = {
    "WT": {
        "label": "A",
        "ax": ax1
    },
    "WT_base": {
        "label": "B",
        "ax": ax2
    },
    "WT_mix": {
        "label": "C",
        "ax": ax3
    },
    "zep2lut2": {
        "label": "D",
        "ax": ax4
    },
    "npq1lut2": {
        "label": "E",
        "ax": ax5
    },
    "npq4lut2": {
        "label": "F",
        "ax": ax6
    },
    "npq4zep2": {
        "label": "G",
        "ax": ax7
    }
}

for GT, df in fig3_res.groupby(level="GT"):
    res = df.droplevel("GT")
    
    ax = axes_settings[GT]["ax"]
    ax.plot(res["tau_Fluo"], lw=3, color="#77ac30")
    
    ax.set_ylim(0, 2)
    ax.set_yticks(np.linspace(0, 1.5, 4))
    ax.set_ylabel(r"$\tau_\mathrm{avg}$")
    
    ax.set_xlim(-1, 20)
    ax.set_xticks(np.linspace(0, 20, 3))
    
    ax.set_title(GT, fontweight="bold", fontstyle="italic")
    ax.text(-0.01, 1.01, axes_settings[GT]["label"], fontweight="bold", transform=ax.transAxes, ha="right", va="bottom", fontsize=12)
    
    if "mix" in GT:
        box = mix_box
    elif "base" in GT:
        box = base_box
    else:
        box = normal_box
    
    for i, (x2, ppfd) in enumerate(box.iterrows()):
        x = ax.get_xlim()[0] if x2 == 0 else box.iloc[i - 1].name
        color = "white" if ppfd.values == 1 else "black"
        
        ax.add_patch(plt.Rectangle(
            (x, 0),
            width=x2 - x,
            height=0.1,
            facecolor=color,
            edgecolor="black"
        ))

plt.tight_layout()

## Figure 4

In [ ]:
def fig4_sim(settings):
    res = Simulator(settings["model"]).simulate_protocol_time_course(
        protocol=settings["protocol"],
        time_points=[1+1, 3+1, 5+1, 20+1]
    ).get_result().unwrap_or_err().get_combined()
    
    res.index = res.index - settings["protocol"].index[0].total_seconds()
    return res[res.index >= 0]

fig4_prtc = make_protocol([(1, {"ppfd": 0}), (20, {"ppfd": 1})])

fig4_settings = {
    "WT": {
        "model": get_lam2026(),
        "protocol": fig4_prtc
    },
    "npq4npq1": {
        "model": create_npq4npq1(get_lam2026()),
        "protocol": fig4_prtc
    },
    "npq4": {
        "model": create_npq4(get_lam2026()),
        "protocol": fig4_prtc
    },
    "npq1": {
        "model": create_npq1(get_lam2026()),
        "protocol": fig4_prtc
    },
    "lut2": {
        "model": create_lut2(get_lam2026()),
        "protocol": fig4_prtc
    },
    "zep2": {
        "model": create_zep2(get_lam2026()),
        "protocol": fig4_prtc
    }

}

fig4_res = dict(parallelise(fig4_sim, list(fig4_settings.items())))
fig4_res = pd.concat(fig4_res.values(), keys=fig4_res.keys())
fig4_res.index.names = ["GT", "time"]

In [ ]:
fig4, axs = plt.subplots(2, 2, figsize=(9, 8))

axes_settings = {
    "npq4npq1": 1,
    "npq4": 2,
    "npq1": 3,
    "lut2": 4,
    "zep2": 5,
    "WT": 6
}

plot_stylings = {
    "NPQ_qI": {"color": "#7e2f8e", "label": "qI"},
    "NPQ_V": {"color": "#edb120", "label": "V"},
    "NPQ_A": {"color": "#d95319", "label": "A"},
    "NPQ_Z_qE": {"color": "#77ac30", "label": "Z (qE)"},
    "NPQ_L": {"color": "#0072bd", "label": "L"},
    "NPQ_Z_qZ": {"color": "#8fed8f", "label": "Z (qZ)"}
}

for GT, df in fig4_res.groupby(level="GT"):
    res = df.droplevel("GT")
    for i, (time, ax) in enumerate(zip(res.index[1:], axs.flat, strict=False)):
        bottom = 0
        for var in plot_stylings:
            ax.bar(axes_settings[GT], res.loc[time][var], bottom=bottom, **plot_stylings[var])
            bottom += res.loc[time][var]
        ax.set_title(f"Time = {int(time)} mins")

custom_handles = [mpatches.Patch(color=vals["color"], label=vals["label"]) for vals in plot_stylings.values()][::-1]

for i, ax in enumerate(axs.flat):
    ax.set_ylim(0, 6)
    ax.set_yticks(np.linspace(0, 6, 7))
    ax.set_ylabel("NPQ contributon")
    ax.set_xticks(list(axes_settings.values()))
    ax.set_xticklabels(axes_settings.keys(), rotation=45, ha="right")
    ax.text(-0.01, 1.05, string.ascii_uppercase[i], transform=ax.transAxes, fontsize=12, fontweight="bold", ha="right")
    ax.legend(handles=custom_handles, loc="upper left")

plt.tight_layout()

## Figure 5

In [ ]:
def fig5_sim(settings):
    m = get_lam2026()
    old_params = m.get_parameter_values()
    changes = settings["changes"]
    
    parameters_scan = {}
    if changes != {}:
        for change in changes:
            if change == "VDE":
                for param in ["k_L_VA", "k_D_VA", "k_L_AZ"]:
                    parameters_scan[param] = [old_params[param] * i for i in changes[change]]
            if change == "ZEP":
                for param in ["k_AV", "k_ZA"]:
                    parameters_scan[param] = [old_params[param] * i for i in changes[change]]
            if change == "PsbS":
                for param in ["kappa_QV", "kappa_QA", "kappa_QZ", "kappa_QL"]:
                    parameters_scan[param] = [old_params[param] * i for i in changes[change]]
    else:
        return Simulator(m).simulate_protocol(settings["protocol"], time_points_per_step=int(1e3)).get_result().unwrap_or_err().get_combined()
    
    return scan.protocol(model=m, to_scan=pd.DataFrame(parameters_scan), protocol=settings["protocol"], time_points_per_step=int(1e3)).combined

fig5_prtc = make_protocol([
    (1, {"ppfd": 0}),
    (5, {"ppfd": 1}),
    (10, {"ppfd": 0}),
    (5, {"ppfd": 1}),
])

fig5_settings = {
    "WT": {
        "changes": {},
        "protocol": fig5_prtc
    },
    "A": {
        "changes": {
            "VDE": [2, 5, 25]
        },
        "protocol": fig5_prtc
    },
    "B": {
        "changes": {
            "ZEP": [2, 5, 25]
        },
        "protocol": fig5_prtc
    },
    "C": {
        "changes": {
            "PsbS": [2, 5, 25]
        },
        "protocol": fig5_prtc
    },
    "D": {
        "changes": {
            "VDE": [1, 10],
            "ZEP": [5, 50],
            "PsbS": [2, 5]
        },
        "protocol": fig5_prtc
    },
}

fig5_res = dict(parallelise(fig5_sim, list(fig5_settings.items())))

In [ ]:
fig5, axs = plt.subplots(nrows=2, ncols=2, figsize=(10, 8))

plot_stylings = {
    "0": {"color": "#ff6161"},
    "1": {"color": "#ffc233"},
    "2": {"color": "#47a3ff"},
}

box = fig5_prtc.copy()
box.index = box.index.total_seconds()
box.index = box.index - box.index[0]

for key, df in fig5_res.items():
    df = df.copy()
    if key == "WT":
        df.index = df.index - fig5_prtc.index[0].total_seconds()
        df = df[df.index >= 0]
        for ax in axs.flat:
            ax.plot(df["tau_Fluo"], color="black", label="WT")
    else:
        if key == "A":
            ax = axs[0, 0]
            labels = [f"{i}x VDE, 1x ZEP, 1x PsbS" for i in [2, 5, 25]]
        elif key == "B":
            ax = axs[0, 1]
            labels = [f"1x VDE, {i}x ZEP, 1x PsbS" for i in [2, 5, 25]]
        elif key == "C":
            ax = axs[1, 0]
            labels = [f"1x VDE, 1x ZEP, {i}x PsbS" for i in [2, 5, 25]]
        elif key == "D":
            ax = axs[1, 1]
            labels = [f"{i}x VDE, {j}x ZEP, {k}x PsbS" for i, j, k in zip([1, 10], [5, 50], [2, 5], strict=False)]
        for n, df2 in df.groupby(level="n"):
            res = df2.droplevel("n")
            res.index = res.index - fig5_prtc.index[0].total_seconds()
            res = res[res.index >= 0]
            ax.plot(res["tau_Fluo"], **plot_stylings.get(str(n), {"color": "gray"}), ls="-.", label=labels[n])
            
        ax.text(-0.01, 1.05, key, fontweight="bold", transform=ax.transAxes, ha="right", va="bottom", fontsize=14)
            
for ax in axs.flat:
    ax.legend()
    
    ax.set_ylim(-0.1, 2)
    ax.set_yticks(np.linspace(0, 2, 5))
    ax.set_ylabel(r"$\tau_\mathrm{avg}$")
    
    ax.set_xlim(-1, 20)
    ax.set_xlabel("Time (min)")
    
    for i, (x2, ppfd) in enumerate(box.iterrows()):
        x = ax.get_xlim()[0] if x2 == 0 else box.iloc[i - 1].name
        color = "white" if ppfd.values == 1 else "black"
        
        ax.add_patch(plt.Rectangle(
            (x, ax.get_ylim()[0]),
            width=x2 - x,
            height=0.1,
            facecolor=color,
            edgecolor="black"
        ))
        
plt.tight_layout()